<a href="https://colab.research.google.com/github/jdmartinev/CVBootcampMCDA/blob/main/notebooks/01_clip_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📎 Workshop CLIP — Notebook 01: Baseline

---

Este es el primer notebook del workshop. Al terminarlo habrás:

- Cargado el modelo CLIP y entendido su arquitectura dual
- Implementado las funciones base para codificar texto e imágenes
- Construido y analizado la matriz de similitud
- Visualizado el espacio de embeddings compartido

**TODOs en este notebook:** 4  
**Siguiente notebook:** `02_image_retrieval.ipynb` — usarás estas funciones como base

> ⚠️ No se requiere GPU. Colab CPU es suficiente.

---
## 0 — Setup

In [ ]:
%%capture
!pip install transformers datasets Pillow ipywidgets pandas scikit-learn

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset

print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

---
## 1 — Cargar CLIP

CLIP (*Contrastive Language-Image Pre-training*, Radford et al. 2021) entrena simultáneamente dos encoders:

- **Image encoder:** Vision Transformer (ViT-B/32) — divide la imagen en patches de 32×32 y los procesa como tokens
- **Text encoder:** Transformer de 12 capas — codifica el texto tokenizado

Ambos encoders proyectan sus salidas a un **espacio compartido de 512 dimensiones**. El entrenamiento contrastivo los alinea: pares imagen-texto correctos quedan cerca, los incorrectos lejos.

```
Imagen ──► ViT-B/32 ──► Linear(768→512) ──► L2 norm ──►┐
                                                          ├── cosine similarity
Texto  ──► Transformer ► Linear(512→512) ──► L2 norm ──►┘
```

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"

print("Cargando modelo CLIP...")
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
clip_model.eval()
print("✅ Listo")

In [ ]:
# Inspeccionar la arquitectura
# Observa: visual_projection (768→512) y text_projection (512→512)
print(clip_model)

In [ ]:
total = sum(p.numel() for p in clip_model.parameters()) / 1e6
vision = sum(p.numel() for p in clip_model.vision_model.parameters()) / 1e6
text   = sum(p.numel() for p in clip_model.text_model.parameters()) / 1e6

print(f"Parámetros totales:          {total:.1f}M")
print(f"  Vision encoder (ViT-B/32): {vision:.1f}M")
print(f"  Text encoder:              {text:.1f}M")

---
## 2 — Cargar datos

Usamos **Flickr30k**: 31K imágenes con 5 captions cada una, estándar para evaluar sistemas de image-text retrieval.

Para este notebook cargamos un subset pequeño de 50 imágenes del split `test` — suficiente para explorar los conceptos sin tiempos de espera.

> En `02_image_retrieval.ipynb` escalaremos a 2000 imágenes para construir el motor de búsqueda completo.

In [ ]:
SEED = 42
DEMO_SIZE = 50

print("Cargando Flickr30k...")
raw = load_dataset("AnyModal/flickr30k", split="test")

import random
random.seed(SEED)
demo_indices = random.sample(range(len(raw)), DEMO_SIZE)

demo_images   = [raw[i]["image"].convert("RGB") for i in demo_indices]
demo_captions = [raw[i]["original_alt_text"] for i in demo_indices]
demo_ids      = [str(raw[i]["img_id"]) for i in demo_indices]

print(f"✅ {DEMO_SIZE} imágenes cargadas")
print(f"\nEjemplo — imagen 0, captions:")
for c in demo_captions[0]:
    print(f"  · {c}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(demo_images[i])
    ax.set_title(demo_captions[i][0][:40] + "...", fontsize=7, wrap=True)
    ax.axis("off")
plt.suptitle("Muestra del dataset (imagen + primer caption)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 3 — Codificar texto

El flujo para obtener un embedding de texto es:

```
str → CLIPProcessor (tokenización) → tensor de token IDs → CLIPTextTransformer → vector (512,) → L2 normalize
```

**Normalización L2:** dividir cada vector por su norma para que quede en la esfera unitaria. Con vectores normalizados, la similitud coseno es simplemente el producto punto:

$$\text{sim}(a, b) = \frac{a \cdot b}{\|a\|\|b\|} = a_{\hat{}} \cdot b_{\hat{}}$$

### ✏️ TODO 1 — `get_text_embeddings`

Implementa la función que convierte una lista de strings en embeddings normalizados.

**Pasos:**
1. Preprocesa con `clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77)` y mueve los tensores al `device` con `.to(device)`.
2. Dentro de `torch.no_grad()`, llama a `model.get_text_features(**inputs)` — devuelve shape `(N, 512)`.
3. Normaliza con `F.normalize(embeddings, p=2, dim=-1)`.
4. Retorna en CPU con `.cpu()`.

In [ ]:
def get_text_embeddings(
    texts: list[str],
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
) -> torch.Tensor:
    """
    Codifica una lista de textos en embeddings CLIP normalizados L2.

    Args:
        texts:     lista de N strings
        model:     CLIPModel en eval mode
        processor: CLIPProcessor correspondiente
        device:    dispositivo de cómputo

    Returns:
        Tensor de shape (N, 512), normalizado L2, en CPU.
    """
    # TODO 1
    raise NotImplementedError


# ── Test ──────────────────────────────────────────────────────────────────────
# Descomenta cuando hayas implementado la función:
#
# embs = get_text_embeddings(["a dog", "a cat", "a rocket"], clip_model, clip_processor, device)
# assert embs.shape == (3, 512), f"Shape incorrecto: {embs.shape}"
# norms = embs.norm(dim=-1)
# assert torch.allclose(norms, torch.ones(3), atol=1e-5), f"No normalizado: {norms}"
# print("✅ TODO 1 OK")

---
## 4 — Codificar imágenes

El flujo para imágenes es análogo al de texto:

```
PIL Image → CLIPProcessor (resize, crop, normalize pixels) → tensor → CLIPVisionTransformer → vector (512,) → L2 normalize
```

El `CLIPProcessor` para imágenes aplica:
- Resize a 224×224
- Center crop
- Normalización de canales RGB con media y std específicos de CLIP

### ✏️ TODO 2 — `get_image_embeddings`

Implementa la función análoga para imágenes. La estructura es idéntica al TODO 1, cambia solo el argumento del processor y el método del modelo.

**Pasos:**
1. Preprocesa con `clip_processor(images=images, return_tensors="pt")` y mueve al `device`.
2. Dentro de `torch.no_grad()`, llama a `model.get_image_features(**inputs)`.
3. Normaliza L2 y retorna en CPU.

In [ ]:
def get_image_embeddings(
    images: list,
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
) -> torch.Tensor:
    """
    Codifica una lista de imágenes PIL en embeddings CLIP normalizados L2.

    Args:
        images:    lista de N PIL Images
        model:     CLIPModel en eval mode
        processor: CLIPProcessor correspondiente
        device:    dispositivo de cómputo

    Returns:
        Tensor de shape (N, 512), normalizado L2, en CPU.
    """
    # TODO 2
    raise NotImplementedError


# ── Test ──────────────────────────────────────────────────────────────────────
# embs = get_image_embeddings(demo_images[:4], clip_model, clip_processor, device)
# assert embs.shape == (4, 512), f"Shape incorrecto: {embs.shape}"
# norms = embs.norm(dim=-1)
# assert torch.allclose(norms, torch.ones(4), atol=1e-5), f"No normalizado: {norms}"
# print("✅ TODO 2 OK")

---
## 5 — Matriz de similitud

Con ambas funciones implementadas podemos calcular la similitud entre **todos los pares** imagen-texto de un batch en una sola operación matricial:

$$S = I \cdot T^\top \in \mathbb{R}^{N \times N}$$

donde $I \in \mathbb{R}^{N \times 512}$ son los embeddings de imagen y $T \in \mathbb{R}^{N \times 512}$ los de texto. El elemento $S_{ij}$ es la similitud coseno entre la imagen $i$ y el texto $j$.

En una alineación perfecta, $S$ sería una matriz identidad: cada imagen más similar a su propio caption que a los demás.

### ✏️ TODO 3 — `compute_similarity_matrix`

Calcula la matriz de similitud coseno entre N imágenes y N textos.

**Pistas:**
- Ambos tensores ya están normalizados L2 → la similitud coseno es el producto punto
- `image_embs @ text_embs.T` da shape `(N, N)` directamente
- Escala por `tau` antes de retornar

In [ ]:
def compute_similarity_matrix(
    image_embs: torch.Tensor,
    text_embs: torch.Tensor,
    tau: float = 1.0,
) -> torch.Tensor:
    """
    Matriz de similitud coseno entre imágenes y textos.

    Args:
        image_embs: Tensor (N, 512) normalizado
        text_embs:  Tensor (N, 512) normalizado
        tau:        temperatura — escala los scores (default 1.0 = sin escala)

    Returns:
        Tensor (N, N) donde S[i,j] = sim(imagen_i, texto_j)
    """
    # TODO 3 — una línea
    raise NotImplementedError


# ── Test ──────────────────────────────────────────────────────────────────────
# a = F.normalize(torch.randn(5, 512), dim=-1)
# b = F.normalize(torch.randn(5, 512), dim=-1)
# S = compute_similarity_matrix(a, b)
# assert S.shape == (5, 5)
# assert S.min() >= -1.01 and S.max() <= 1.01
# S_id = compute_similarity_matrix(a, a)
# assert torch.allclose(S_id.diagonal(), torch.ones(5), atol=1e-5)
# print("✅ TODO 3 OK")

### 5.2 — Visualizar la matriz de similitud

Computamos la matriz de similitud para las primeras 15 imágenes del demo y sus captions, y la visualizamos como heatmap. **En una alineación perfecta esperaríamos una diagonal brillante.**

In [ ]:
N_VIZ = 15

img_embs  = get_image_embeddings(demo_images[:N_VIZ], clip_model, clip_processor, device)
text_embs = get_text_embeddings([demo_captions[i][0] for i in range(N_VIZ)],
                                 clip_model, clip_processor, device)

S = compute_similarity_matrix(img_embs, text_embs).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S, cmap="RdYlGn", vmin=-0.3, vmax=1.0)
plt.colorbar(im, ax=ax, label="Similitud coseno")
ax.set_xlabel("Texto (caption)"); ax.set_ylabel("Imagen")
ax.set_xticks(range(N_VIZ)); ax.set_yticks(range(N_VIZ))
ax.set_xticklabels(range(N_VIZ)); ax.set_yticklabels(range(N_VIZ))
for i in range(N_VIZ):
    ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                                fill=False, edgecolor="black", linewidth=2))
ax.set_title("Matriz de similitud CLIP (imagen × texto)\nDiagonal = pares correctos", fontsize=12)
plt.tight_layout()
plt.show()

diag_mean    = np.diag(S).mean()
offdiag_mean = (S.sum() - np.diag(S).sum()) / (N_VIZ * (N_VIZ - 1))
print(f"Similitud media diagonal (pares correctos): {diag_mean:.3f}")
print(f"Similitud media fuera diagonal:             {offdiag_mean:.3f}")
print(f"Gap diagonal - off-diagonal:               {diag_mean - offdiag_mean:.3f}")

---
## 6 — Similitud query → corpus (1-a-N)

En retrieval la operación habitual no es N×N sino **1×N**: dado un único query, calcular su similitud contra todos los elementos del corpus.

$$\text{scores} = C \cdot q^\top \in \mathbb{R}^{N}$$

donde $C \in \mathbb{R}^{N \times 512}$ es la matriz del corpus y $q \in \mathbb{R}^{1 \times 512}$ el embedding del query.

### ✏️ TODO 4 — `compute_scores`

Similitud coseno entre un único query embedding y una matriz de corpus.

**Pistas:**
- `query_emb` shape `(1, 512)`, `corpus_emb` shape `(N, 512)`
- Resultado shape `(N,)` — un score por elemento del corpus
- Recuerda hacer `.squeeze()` al final para eliminar la dimensión de batch

In [ ]:
def compute_scores(
    query_emb: torch.Tensor,
    corpus_emb: torch.Tensor,
) -> torch.Tensor:
    """
    Similitud coseno entre un query y cada vector del corpus.

    Args:
        query_emb:  Tensor (1, 512) normalizado
        corpus_emb: Tensor (N, 512) normalizado

    Returns:
        Tensor (N,) con scores ∈ [-1, 1]
    """
    # TODO 4 — una línea
    raise NotImplementedError


# ── Test ──────────────────────────────────────────────────────────────────────
# q = F.normalize(torch.randn(1, 512), dim=-1)
# C = F.normalize(torch.randn(50, 512), dim=-1)
# scores = compute_scores(q, C)
# assert scores.shape == (50,), f"Shape incorrecto: {scores.shape}"
# assert scores.min() >= -1.01 and scores.max() <= 1.01
# C[7] = q.squeeze()
# scores = compute_scores(q, C)
# assert abs(scores[7].item() - 1.0) < 1e-5
# print("✅ TODO 4 OK")

### 6.2 — Query de texto → imágenes

Calculamos los scores de similitud entre un texto y todas las imágenes del corpus, y mostramos las top-3 junto con su score.

In [ ]:
corpus_embs = get_image_embeddings(demo_images, clip_model, clip_processor, device)

query_text  = "a group of people at a sports event"
text_q_emb  = get_text_embeddings([query_text], clip_model, clip_processor, device)
text_scores = compute_scores(text_q_emb, corpus_embs).numpy()
top3_idx    = text_scores.argsort()[-3:][::-1]

fig = plt.figure(figsize=(14, 6))
ax_bar = fig.add_axes([0.05, 0.62, 0.90, 0.30])
ax_bar.bar(range(DEMO_SIZE), text_scores, color="steelblue", alpha=0.6)
for t in top3_idx:
    ax_bar.bar(t, text_scores[t], color="tomato")
ax_bar.set_ylabel("Similitud coseno")
ax_bar.set_title(f'Query: "{query_text}"', fontsize=11, fontweight="bold")
ax_bar.set_xticks([])
for rank, idx in enumerate(top3_idx):
    ax = fig.add_axes([0.05 + rank * 0.32, 0.02, 0.28, 0.55])
    ax.imshow(demo_images[idx])
    ax.set_title(f"#{rank+1}  sim={text_scores[idx]:.3f}\n{demo_captions[idx][0][:45]}...", fontsize=8)
    ax.axis("off")
plt.suptitle("Top-3 resultados — query de TEXTO", fontsize=12, y=1.01)
plt.show()

### 6.3 — Query de imagen → imágenes similares

Ahora el query es una imagen. El pipeline es **idéntico** — solo cambia que el embedding del query viene de `get_image_embeddings` en lugar de `get_text_embeddings`. Esto es posible porque ambas modalidades viven en el mismo espacio de 512 dimensiones.

In [ ]:
query_idx   = 0  # cambia este índice para explorar
query_image = demo_images[query_idx]

img_q_emb  = get_image_embeddings([query_image], clip_model, clip_processor, device)
img_scores = compute_scores(img_q_emb, corpus_embs).numpy()
img_scores[query_idx] = -1.0
top3_idx = img_scores.argsort()[-3:][::-1]

fig = plt.figure(figsize=(18, 5))
ax_q = fig.add_axes([0.01, 0.08, 0.20, 0.84])
ax_q.imshow(query_image)
ax_q.set_title(f"QUERY (idx={query_idx})\n{demo_captions[query_idx][0][:45]}...",
               fontsize=8, color="crimson", fontweight="bold")
ax_q.axis("off")
fig.text(0.235, 0.50, "→", fontsize=28, ha="center", va="center", color="gray")
for rank, idx in enumerate(top3_idx):
    ax = fig.add_axes([0.27 + rank * 0.245, 0.08, 0.22, 0.84])
    ax.imshow(demo_images[idx])
    ax.set_title(f"#{rank+1}  sim={img_scores[idx]:.3f}\n{demo_captions[idx][0][:45]}...", fontsize=8)
    ax.axis("off")
plt.suptitle("Top-3 resultados — query de IMAGEN", fontsize=12)
plt.show()
print("💬 Cambia query_idx y vuelve a ejecutar para explorar.")

---
## ✅ Checkpoint

Antes de pasar al siguiente notebook, verifica que tienes las cuatro funciones implementadas y testeadas:

| Función | Shape entrada | Shape salida | TODO |
|---------|--------------|-------------|------|
| `get_text_embeddings` | `list[str]` | `(N, 512)` | 1 |
| `get_image_embeddings` | `list[PIL]` | `(N, 512)` | 2 |
| `compute_similarity_matrix` | `(N,512), (N,512)` | `(N, N)` | 3 |
| `compute_scores` | `(1,512), (N,512)` | `(N,)` | 4 |

En `02_image_retrieval.ipynb` estas cuatro funciones son la base de todo el motor de búsqueda. **Cópialas en la celda de setup del siguiente notebook.**

In [ ]:
print("Ejecutando test integral...")

t_embs = get_text_embeddings(["a dog", "a cat"], clip_model, clip_processor, device)
i_embs = get_image_embeddings(demo_images[:2], clip_model, clip_processor, device)
S      = compute_similarity_matrix(i_embs, t_embs)
sc     = compute_scores(t_embs[:1], i_embs)

assert t_embs.shape == (2, 512)
assert i_embs.shape == (2, 512)
assert S.shape == (2, 2)
assert sc.shape == (2,)

print("✅ Todas las funciones OK — listo para el notebook 02")